# SimpleApp using LangChain + Hugging Face

In [ ]:
# Uncomment if running in fresh environment
# !pip install langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")


In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
docs = loader.load()

print("Docs loaded:", len(docs))
print(docs[0].page_content[:500])


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = text_splitter.split_documents(docs)

print("Chunks:", len(documents))
print(documents[0].page_content[:500])


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()


In [ ]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"]
)

prompt = ChatPromptTemplate.from_template("""
Answer ONLY using the context below:

<context>
{context}
</context>

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm, prompt)


In [ ]:
from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)


In [ ]:
response = retrieval_chain.invoke({
    "input": "LangSmith has two usage limits"
})

print("ANSWER:\n", response["answer"])


In [ ]:
print("\n--- Retrieved Context ---\n")
for i, doc in enumerate(response["context"][:3]):
    print(f"Chunk {i+1}:")
    print(doc.page_content[:300])
    print("-"*50)
